# Exploración ICS e IPM

Carga de shapefiles desde el repositorio, visualización en mapa y análisis inicial del Índice de Condición Social (ICS) y el Índice de Pobreza Multidimensional (IPM) a nivel de manzana.


In [ ]:
# @title 1. Montar Google Drive (opcional, solo en Colab)
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive montado')
else:
    print('Ejecutando localmente')


In [ ]:
# @title 2. Instalar dependencias
!pip install geopandas matplotlib mapclassify pyarrow openpyxl -q


In [ ]:
# @title 3. Clonar el repositorio (solo si es necesario)
import os
import shutil

REPO_URL = 'https://github.com/j0rg3c45/Pobreza_multidimensional_y_condicion_social.git'
REPO_DIR = 'Pobreza_multidimensional_y_condicion_social'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}
else:
    %cd {REPO_DIR}
    !git pull
    %cd ..
print('Repositorio listo')


In [ ]:
# @title 4. Importar librerías
import zipfile
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.patches as mpatches
from pathlib import Path

print('Librerías importadas')


In [ ]:
# @title 5. Definir rutas y descomprimir
BASE_DIR = REPO_DIR if os.path.exists(REPO_DIR) else '.'
DATA_DIR = os.path.join(BASE_DIR, 'indice_Pobreza', 'data')
EXCEL_PATH = os.path.join(BASE_DIR, 'IPM - Variables (incidencias).xlsx')

def extract_zip(zip_name):
    zip_path = os.path.join(DATA_DIR, zip_name)
    out_dir = os.path.join(DATA_DIR, zip_name.replace('.zip', ''))
    if not os.path.exists(out_dir):
        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extractall(out_dir)
        print(f'Extraído: {zip_name} → {out_dir}')
    else:
        print(f'Ya existe: {out_dir}')
    return out_dir

ics_dir = extract_zip('ICS.zip')
ipm_dir = extract_zip('IPM.zip')
print(f'Excel existe: {os.path.exists(EXCEL_PATH)}')


In [ ]:
# @title 6. Cargar shapefiles
gdf_ics = gpd.read_file(os.path.join(ics_dir, 'Mzn_ics.shp'))
gdf_ipm = gpd.read_file(os.path.join(ipm_dir, 'Mzn_ipm.shp'))

print(f'ICS: {gdf_ics.shape[0]} geometrías, {gdf_ics.shape[1]} columnas')
print(f'IPM: {gdf_ipm.shape[0]} geometrías, {gdf_ipm.shape[1]} columnas')


In [ ]:
# @title 7. Ver columnas disponibles
print('=== Columnas ICS ===')
print(gdf_ics.columns.tolist())
print()
print('=== Columnas IPM ===')
print(gdf_ipm.columns.tolist())


In [ ]:
# @title 8. Vista previa de los datos
print('=== ICS (primeras 5 filas) ===')
display(gdf_ics.head())
print()
print('=== IPM (primeras 5 filas) ===')
display(gdf_ipm.head())


In [ ]:
# @title 9. Sistema de referencia de coordenadas (CRS)
print(f'ICS CRS: {gdf_ics.crs}')
print(f'IPM CRS: {gdf_ipm.crs}')


In [ ]:
# @title 10. Mapa base — Manzanas ICS
fig, ax = plt.subplots(1, 1, figsize=(14, 12))
gdf_ics.plot(ax=ax, color='#e0e0e0', edgecolor='#999999', linewidth=0.3, alpha=0.7)
ax.set_title('Manzanas — ICS (Índice de Condición Social)', fontsize=14, fontweight='bold')
ax.set_axis_off()
plt.tight_layout()
plt.show()


In [ ]:
# @title 11. Mapa base — Manzanas IPM
fig, ax = plt.subplots(1, 1, figsize=(14, 12))
gdf_ipm.plot(ax=ax, color='#e0e0e0', edgecolor='#999999', linewidth=0.3, alpha=0.7)
ax.set_title('Manzanas — IPM (Índice de Pobreza Multidimensional)', fontsize=14, fontweight='bold')
ax.set_axis_off()
plt.tight_layout()
plt.show()


In [ ]:
# @title 12. Identificar columna de ICS para clasificación
# Buscar columna numérica que contenga el valor del ICS
ics_cols = [c for c in gdf_ics.columns if gdf_ics[c].dtype in ['float64', 'int64']]
print('Columnas numéricas en ICS:', ics_cols)

# Mostrar distribución de las primeras columnas numéricas
for col in ics_cols[:5]:
    print(f'\n{col}:')
    print(gdf_ics[col].describe())


In [ ]:
# @title 13. Mapa temático — ICS (si existe columna de valor)
# Identificar columna candidata para mapeo
num_cols_ics = [c for c in gdf_ics.columns if gdf_ics[c].dtype in ['float64', 'int64'] and c.lower() not in ['objectid', 'fid', 'id', 'shape_leng', 'shape_area', 'shape_len', 'shape_le']]

if num_cols_ics:
    col = num_cols_ics[0]
    fig, ax = plt.subplots(1, 1, figsize=(14, 12))
    gdf_ics.plot(column=col, ax=ax, legend=True,
                 cmap='viridis', edgecolor='white', linewidth=0.2,
                 legend_kwds={'label': col, 'shrink': 0.6})
    ax.set_title(f'ICS — {col}', fontsize=14, fontweight='bold')
    ax.set_axis_off()
    plt.tight_layout()
    plt.show()
else:
    print('No se encontró columna numérica para mapear en ICS')


In [ ]:
# @title 14. Mapa temático — IPM
num_cols_ipm = [c for c in gdf_ipm.columns if gdf_ipm[c].dtype in ['float64', 'int64'] and c.lower() not in ['objectid', 'fid', 'id', 'shape_leng', 'shape_area', 'shape_len', 'shape_le']]

if num_cols_ipm:
    col = num_cols_ipm[0]
    fig, ax = plt.subplots(1, 1, figsize=(14, 12))
    gdf_ipm.plot(column=col, ax=ax, legend=True,
                 cmap='RdYlGn_r', edgecolor='white', linewidth=0.2,
                 legend_kwds={'label': col, 'shrink': 0.6})
    ax.set_title(f'IPM — {col}', fontsize=14, fontweight='bold')
    ax.set_axis_off()
    plt.tight_layout()
    plt.show()
else:
    print('No se encontró columna numérica para mapear en IPM')


In [ ]:
# @title 15. Mapas lado a lado
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 14))

# ICS
if num_cols_ics:
    gdf_ics.plot(column=num_cols_ics[0], ax=ax1, legend=True,
                 cmap='viridis', edgecolor='white', linewidth=0.2,
                 legend_kwds={'label': num_cols_ics[0], 'shrink': 0.5})
    ax1.set_title('ICS', fontsize=13, fontweight='bold')
else:
    gdf_ics.plot(ax=ax1, color='#e0e0e0', edgecolor='#999999', linewidth=0.2)
    ax1.set_title('ICS (sin datos)', fontsize=13, fontweight='bold')
ax1.set_axis_off()

# IPM
if num_cols_ipm:
    gdf_ipm.plot(column=num_cols_ipm[0], ax=ax2, legend=True,
                 cmap='RdYlGn_r', edgecolor='white', linewidth=0.2,
                 legend_kwds={'label': num_cols_ipm[0], 'shrink': 0.5})
    ax2.set_title('IPM', fontsize=13, fontweight='bold')
else:
    gdf_ipm.plot(ax=ax2, color='#e0e0e0', edgecolor='#999999', linewidth=0.2)
    ax2.set_title('IPM (sin datos)', fontsize=13, fontweight='bold')
ax2.set_axis_off()

plt.suptitle('Comparación ICS vs IPM por Manzana', fontsize=16, fontweight='bold', y=0.92)
plt.tight_layout()
plt.show()


In [ ]:
# @title 16. Estadísticas descriptivas
print('=== ICS ===')
if num_cols_ics:
    display(gdf_ics[num_cols_ics].describe())

print('\n=== IPM (shapefile) ===')
if num_cols_ipm:
    display(gdf_ipm[num_cols_ipm].describe())

# Diccionario de variables IPM
diccionario = {
    'analf_': 'Analfabetismo',
    'bajo_': 'Bajo logro educativo',
    'infancia_': 'Barreras primera infancia',
    'inasis_': 'Inasistencia escolar',
    'rezago_': 'Rezago escolar',
    'trab_infan_': 'Trabajo infantil',
    'depen_': 'Dependencia económica',
    'infor_': 'Informalidad',
    'salud_': 'Barreras de salud',
    'asegu_': 'Sin aseguramiento en salud',
    'haci_': 'Hacinamiento crítico',
    'pared_': 'Paredes precarias',
    'excre_': 'Eliminación inadecuada de excretas',
    'pisos_': 'Pisos precarios',
    'agua_': 'Sin acceso a agua mejorada'
}

# Cargar variables IPM desde Excel
xls = pd.ExcelFile(EXCEL_PATH)
sheets = [s for s in xls.sheet_names if s != 'Diccionario']
df_ipm_vars = None
for s in sheets:
    df_var = pd.read_excel(xls, s)
    col_name = df_var.columns[1]
    df_var = df_var.rename(columns={col_name: col_name + '_val'})
    df_var.columns = ['cod_mzn', col_name + '_val']
    if df_ipm_vars is None:
        df_ipm_vars = df_var
    else:
        df_ipm_vars = df_ipm_vars.merge(df_var, on='cod_mzn', how='outer')

val_cols = [c for c in df_ipm_vars.columns if c != 'cod_mzn']
print(f'Cargadas {len(val_cols)} variables IPM para {df_ipm_vars.shape[0]} manzanas')

print('\n=== Variables IPM (Excel) - Estadísticas ===')
display(df_ipm_vars[val_cols].describe().round(2))

# Top 5
means = df_ipm_vars[val_cols].mean().sort_values(ascending=False)
print('\n--- Top 5 variables IPM más críticas ---')
for var in means.head().index:
    label = diccionario.get(var.replace('_val',''), var)
    print(f'  {label}: {means[var]:.2f}%')
print(f'\nVariable más crítica: {diccionario.get(means.index[0].replace("_val",""), means.index[0])} ({means.iloc[0]:.2f}%)')


## Enriquecimiento geoespacial: Comunas y zonas de interés


In [ ]:
# @title 17. Cargar GeoJSONs auxiliares (comunas, Barrio Obrero, Roosevelt)
gdf_comunas = gpd.read_file(os.path.join(DATA_DIR, 'geojson_comunas', 'Comunas.geojson'))
print(f'Comunas: {gdf_comunas.shape[0]} geometr\u00edas, {gdf_comunas.shape[1]} columnas')
print(f'Columnas: {gdf_comunas.columns.tolist()}')
display(gdf_comunas[['comuna', 'nombre']])

gdf_obrero = gpd.read_file(os.path.join(DATA_DIR, 'Geojson_Barrio_Obrero', 'Geojson_Barrio_Obrero.geojson'))
print(f'Barrio Obrero: {gdf_obrero.shape[0]} geometr\u00edas')

gdf_roosevelt = gpd.read_file(os.path.join(DATA_DIR, 'Geojson_Roosevelt', 'tramos_Roosevelt_Buffer_100.geojson'))
print(f'Roosevelt: {gdf_roosevelt.shape[0]} geometr\u00edas')


In [ ]:
# @title 18. Unificar CRS y asignar comuna por spatial join
crs_ref = gdf_ics.crs
print(f'CRS de referencia (ICS): {crs_ref}')

gdf_comunas_proj = gdf_comunas.to_crs(crs_ref)
gdf_ipm_proj = gdf_ipm.to_crs(crs_ref)

gdf_ics_con_comuna = gpd.sjoin(gdf_ics, gdf_comunas_proj[['comuna', 'nombre', 'geometry']], how='left', predicate='intersects')
gdf_ipm_con_comuna = gpd.sjoin(gdf_ipm_proj, gdf_comunas_proj[['comuna', 'nombre', 'geometry']], how='left', predicate='intersects')

comunas_ics = gdf_ics_con_comuna['comuna'].notna().sum()
comunas_ipm = gdf_ipm_con_comuna['comuna'].notna().sum()
print(f'Manzanas ICS con comuna asignada: {comunas_ics} / {len(gdf_ics_con_comuna)}')
print(f'Manzanas IPM con comuna asignada: {comunas_ipm} / {len(gdf_ipm_con_comuna)}')


In [ ]:
# @title 19. Vista previa con comuna asignada
print('=== ICS con comuna ===')
display(gdf_ics_con_comuna.drop(columns='index_right', errors='ignore').head())

print('\n=== IPM con comuna ===')
display(gdf_ipm_con_comuna.drop(columns='index_right', errors='ignore').head())


In [ ]:
# @title 20. Estadísticas de ICS, IPM y variables por comuna
print('=== ICS promedio por comuna ===')
ics_por_comuna = gdf_ics_con_comuna.groupby('nombre').agg(
    manzanas=('cod_dane_a', 'count'),
    ics_promedio=('ics', 'mean'),
    ics_min=('ics', 'min'),
    ics_max=('ics', 'max')
).round(2).sort_values('ics_promedio', ascending=False)
display(ics_por_comuna)

print('=== IPM global promedio por comuna ===')
ipm_por_comuna = gdf_ipm_con_comuna.groupby('nombre').agg(
    manzanas=('COD_DANE', 'count'),
    ipm_promedio=('ipm', 'mean'),
    ipm_min=('ipm', 'min'),
    ipm_max=('ipm', 'max')
).round(2).sort_values('ipm_promedio')
display(ipm_por_comuna)

# Merge Excel vars con ICS para obtener geometrías
# Clave: quitar los 2 dígitos de comuna del código Excel
df_ipm_vars['COD_MZN'] = df_ipm_vars['cod_mzn'].astype(str).str[:6] + df_ipm_vars['cod_mzn'].astype(str).str[8:]

df_vars_geom = gdf_ics[['COD_MZN', 'geometry']].merge(
    df_ipm_vars, on='COD_MZN', how='inner')
gdf_vars = gpd.GeoDataFrame(df_vars_geom, crs=gdf_ics.crs)
print(f'\nMerge Excel+ICS: {len(gdf_vars)}/{len(df_ipm_vars)} manzanas con geometría')

# Spatial join con comunas
gdf_vars_comuna = gpd.sjoin(gdf_vars, gdf_comunas_proj[['comuna', 'nombre', 'geometry']],
                             how='left', predicate='intersects')

# Top 3 variables IPM por comuna
print('\n=== Top 3 variables IPM por comuna ===')
comuna_stats = gdf_vars_comuna.groupby('nombre')[val_cols].mean().round(2)
top3_vars = means.head(3).index.tolist()
for var in top3_vars:
    label = diccionario.get(var.replace('_val',''), var)
    df_show = comuna_stats[var].sort_values(ascending=False).head(5).to_frame(label)
    print(f'\n{label} (top 5 comunas):')
    display(df_show)


In [ ]:
# @title 21. Identificar manzanas en zonas especiales
# Manejar CRS de Barrio Obrero (ESRI:103599)
if gdf_obrero.crs is None:
    gdf_obrero = gdf_obrero.set_crs('ESRI:103599')
gdf_obrero_proj = gdf_obrero.to_crs(crs_ref)

gdf_roosevelt_proj = gdf_roosevelt.to_crs(crs_ref)

# Identificar manzanas dentro de cada zona
gdf_ics['en_barrio_obrero'] = ~gpd.sjoin(gdf_ics, gdf_obrero_proj, how='left', predicate='intersects')['index_right'].isna()
gdf_ics['en_roosevelt'] = ~gpd.sjoin(gdf_ics, gdf_roosevelt_proj, how='left', predicate='intersects')['index_right'].isna()

en_obrero = gdf_ics['en_barrio_obrero'].sum()
en_roosevelt = gdf_ics['en_roosevelt'].sum()
print(f'Manzanas en Barrio Obrero: {en_obrero}')
print(f'Manzanas en Roosevelt: {en_roosevelt}')

if en_obrero > 0:
    print('=== ICS en Barrio Obrero ===')
    display(gdf_ics[gdf_ics['en_barrio_obrero']][['COD_MZN', 'ics']])

if en_roosevelt > 0:
    print('=== ICS en Roosevelt ===')
    display(gdf_ics[gdf_ics['en_roosevelt']][['COD_MZN', 'ics']])


In [ ]:
# @title 22. Mapa - Manzanas por comuna con zonas de inter\u00e9s
fig, ax = plt.subplots(1, 1, figsize=(14, 12))

gdf_ics_con_comuna.plot(column='nombre', ax=ax, legend=True,
                         cmap='tab20', edgecolor='white', linewidth=0.1, alpha=0.7,
                         legend_kwds={'title': 'Comuna', 'loc': 'upper left', 'fontsize': 8})

gdf_comunas_proj.boundary.plot(ax=ax, color='black', linewidth=1.2, alpha=0.8)
gdf_roosevelt_proj.boundary.plot(ax=ax, color='red', linewidth=2, label='Roosevelt (buffer 100m)')
gdf_obrero_proj.boundary.plot(ax=ax, color='blue', linewidth=2, label='Barrio Obrero')

ax.set_title('Manzanas por comuna con zonas de inter\u00e9s', fontsize=14, fontweight='bold')
ax.set_axis_off()
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# @title 23. Mapa - Variables IPM más críticas
# Mapa de las 3 variables IPM con mayor incidencia
top3_vars = means.head(3).index.tolist()
fig, axes = plt.subplots(1, 3, figsize=(24, 8))

for ax, var in zip(axes, top3_vars):
    label = diccionario.get(var.replace('_val',''), var)
    gdf_vars.plot(column=var, ax=ax, legend=True,
                 cmap='RdYlGn_r', edgecolor='white', linewidth=0.1,
                 legend_kwds={'shrink': 0.5},
                 vmin=0, vmax=100)
    gdf_comunas_proj.boundary.plot(ax=ax, color='black', linewidth=0.8, alpha=0.5)
    ax.set_title(label, fontsize=12, fontweight='bold')
    ax.set_axis_off()

plt.suptitle('Variables IPM más críticas en Cali', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# @title 24. Filtrar variables IPM por zona
# Spatial join de variables IPM con cada zona de interés
gdf_vars_obrero = gpd.sjoin(gdf_vars, gdf_obrero_proj, how='inner', predicate='intersects')
gdf_vars_roosevelt = gpd.sjoin(gdf_vars, gdf_roosevelt_proj, how='inner', predicate='intersects')

print(f'Manzanas con vars IPM en Barrio Obrero: {len(gdf_vars_obrero)}')
print(f'Manzanas con vars IPM en Roosevelt: {len(gdf_vars_roosevelt)}')

# Promedio por zona
means_obrero = gdf_vars_obrero[val_cols].mean().sort_values(ascending=False)
means_roosevelt = gdf_vars_roosevelt[val_cols].mean().sort_values(ascending=False)

print('\n--- Top 5 Barrio Obrero ---')
for v in means_obrero.head().index:
    lbl = diccionario.get(v.replace('_val',''), v)
    print(f'  {lbl}: {means_obrero[v]:.2f}%')

print('\n--- Top 5 Roosevelt ---')
for v in means_roosevelt.head().index:
    lbl = diccionario.get(v.replace('_val',''), v)
    print(f'  {lbl}: {means_roosevelt[v]:.2f}%')


In [ ]:
# @title 25. Gráfico - Comparativa variables IPM por zona
# Top 5 variables combinando ambas zonas
top5_vars = (means_obrero + means_roosevelt).nlargest(5).index.tolist()
top5_labels = [diccionario.get(v.replace('_val',''), v) for v in top5_vars]

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Barrio Obrero
vals_obrero = [means_obrero[v] for v in top5_vars]
bars1 = axes[0].barh(top5_labels, vals_obrero, color='steelblue', edgecolor='white')
for bar, v in zip(bars1, vals_obrero):
    axes[0].text(v + 0.5, bar.get_y() + bar.get_height()/2, f'{v:.1f}%',
                 va='center', fontsize=9)
axes[0].set_title('Barrio Obrero - Top 5 privaciones IPM', fontweight='bold')
axes[0].set_xlabel('Incidencia promedio (%)')

# Roosevelt
vals_roosevelt = [means_roosevelt[v] for v in top5_vars]
bars2 = axes[1].barh(top5_labels, vals_roosevelt, color='coral', edgecolor='white')
for bar, v in zip(bars2, vals_roosevelt):
    axes[1].text(v + 0.5, bar.get_y() + bar.get_height()/2, f'{v:.1f}%',
                 va='center', fontsize=9)
axes[1].set_title('Roosevelt - Top 5 privaciones IPM', fontweight='bold')
axes[1].set_xlabel('Incidencia promedio (%)')

plt.suptitle('Comparativa de privaciones IPM por zona', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


## Análisis comparativo: Zonas de interés (ICS e IPM)


In [ ]:
# @title 26. Filtrar ICS para cada zona (spatial join)
gdf_obrero_filt = gpd.sjoin(gdf_ics, gdf_obrero_proj, how='inner', predicate='intersects')
gdf_roosevelt_filt = gpd.sjoin(gdf_ics, gdf_roosevelt_proj, how='inner', predicate='intersects')
print(f'Manzanas ICS en Barrio Obrero: {len(gdf_obrero_filt)}')
print(f'Manzanas ICS en Roosevelt: {len(gdf_roosevelt_filt)}')

if len(gdf_obrero_filt) > 0:
    gdf_obrero_filt = gpd.sjoin(gdf_obrero_filt, gdf_comunas_proj[['comuna', 'nombre', 'geometry']],
                                 how='left', predicate='intersects')
if len(gdf_roosevelt_filt) > 0:
    gdf_roosevelt_filt = gpd.sjoin(gdf_roosevelt_filt, gdf_comunas_proj[['comuna', 'nombre', 'geometry']],
                                   how='left', predicate='intersects')


In [ ]:
# @title 27. Filtrar IPM para cada zona (spatial join)
gdf_ipm_obrero = gpd.sjoin(gdf_ipm_proj, gdf_obrero_proj, how='inner', predicate='intersects')
gdf_ipm_roosevelt = gpd.sjoin(gdf_ipm_proj, gdf_roosevelt_proj, how='inner', predicate='intersects')

print(f'Manzanas IPM en Barrio Obrero: {len(gdf_ipm_obrero)}')
print(f'Manzanas IPM en Roosevelt: {len(gdf_ipm_roosevelt)}')

if len(gdf_ipm_obrero) > 0:
    gdf_ipm_obrero = gpd.sjoin(gdf_ipm_obrero, gdf_comunas_proj[['comuna', 'nombre', 'geometry']],
                                how='left', predicate='intersects')
if len(gdf_ipm_roosevelt) > 0:
    gdf_ipm_roosevelt = gpd.sjoin(gdf_ipm_roosevelt, gdf_comunas_proj[['comuna', 'nombre', 'geometry']],
                                  how='left', predicate='intersects')


In [ ]:
# @title 28. Estadísticas comparativas ICS, IPM global y vars
print('=== COMPARATIVA ICS ===')
comp_ics = pd.DataFrame({
    'Ciudad': gdf_ics['ics'].describe(),
    'Barrio Obrero': gdf_obrero_filt['ics'].describe() if len(gdf_obrero_filt) > 0 else {},
    'Roosevelt': gdf_roosevelt_filt['ics'].describe() if len(gdf_roosevelt_filt) > 0 else {}
}).round(2)
display(comp_ics)

print('\n=== COMPARATIVA IPM global ===')
comp_ipm = pd.DataFrame({
    'Ciudad': gdf_ipm['ipm'].describe(),
    'Barrio Obrero': gdf_ipm_obrero['ipm'].describe() if len(gdf_ipm_obrero) > 0 else {},
    'Roosevelt': gdf_ipm_roosevelt['ipm'].describe() if len(gdf_ipm_roosevelt) > 0 else {}
}).round(2)
display(comp_ipm)

print('\n=== COMPARATIVA variables IPM (promedio) ===')
comp_vars = pd.DataFrame({
    'Barrio Obrero': means_obrero if len(gdf_vars_obrero) > 0 else pd.Series(dtype=float),
    'Roosevelt': means_roosevelt if len(gdf_vars_roosevelt) > 0 else pd.Series(dtype=float)
}).round(2)
if len(comp_vars) > 0:
    comp_vars.index = [diccionario.get(v.replace('_val',''), v) for v in comp_vars.index]
    comp_vars['Diferencia (Roo - BO)'] = (comp_vars['Roosevelt'] - comp_vars['Barrio Obrero']).round(2)
    display(comp_vars.style.background_gradient(cmap='RdYlGn_r', axis=None))
else:
    print('Sin datos de variables IPM por zona')


In [ ]:
# @title 29. Gráficos comparativos ICS e IPM por zona
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# --- Histograma ICS ---
axes[0, 0].hist(gdf_ics['ics'].dropna(), bins=50, alpha=0.4, label='Ciudad', color='gray', density=True)
if len(gdf_obrero_filt) > 0:
    axes[0, 0].hist(gdf_obrero_filt['ics'].dropna(), bins=15, alpha=0.7,
                     label='Barrio Obrero', color='blue', density=True)
if len(gdf_roosevelt_filt) > 0:
    axes[0, 0].hist(gdf_roosevelt_filt['ics'].dropna(), bins=15, alpha=0.7,
                     label='Roosevelt', color='red', density=True)
axes[0, 0].set_title('Distribución ICS', fontweight='bold')
axes[0, 0].set_xlabel('Índice de Condición Social')
axes[0, 0].legend()

# --- Boxplot ICS ---
datos_ics = [gdf_ics['ics'].dropna()]; labels_ics = ['Ciudad']
if len(gdf_obrero_filt) > 0:
    datos_ics.append(gdf_obrero_filt['ics'].dropna()); labels_ics.append('Barrio Obrero')
if len(gdf_roosevelt_filt) > 0:
    datos_ics.append(gdf_roosevelt_filt['ics'].dropna()); labels_ics.append('Roosevelt')
bp1 = axes[0, 1].boxplot(datos_ics, labels=labels_ics, patch_artist=True, showmeans=True)
axes[0, 1].set_title('Comparación ICS', fontweight='bold')
axes[0, 1].set_ylabel('ICS'); axes[0, 1].grid(axis='y', alpha=0.3)

# --- Histograma IPM ---
axes[1, 0].hist(gdf_ipm['ipm'].dropna(), bins=50, alpha=0.4, label='Ciudad', color='gray', density=True)
if len(gdf_ipm_obrero) > 0:
    axes[1, 0].hist(gdf_ipm_obrero['ipm'].dropna(), bins=15, alpha=0.7,
                     label='Barrio Obrero', color='blue', density=True)
if len(gdf_ipm_roosevelt) > 0:
    axes[1, 0].hist(gdf_ipm_roosevelt['ipm'].dropna(), bins=15, alpha=0.7,
                     label='Roosevelt', color='red', density=True)
axes[1, 0].set_title('Distribución IPM', fontweight='bold')
axes[1, 0].set_xlabel('Índice de Pobreza Multidimensional')
axes[1, 0].legend()

# --- Boxplot IPM ---
datos_ipm = [gdf_ipm['ipm'].dropna()]; labels_ipm = ['Ciudad']
if len(gdf_ipm_obrero) > 0:
    datos_ipm.append(gdf_ipm_obrero['ipm'].dropna()); labels_ipm.append('Barrio Obrero')
if len(gdf_ipm_roosevelt) > 0:
    datos_ipm.append(gdf_ipm_roosevelt['ipm'].dropna()); labels_ipm.append('Roosevelt')
bp2 = axes[1, 1].boxplot(datos_ipm, labels=labels_ipm, patch_artist=True, showmeans=True)
axes[1, 1].set_title('Comparación IPM', fontweight='bold')
axes[1, 1].set_ylabel('IPM'); axes[1, 1].grid(axis='y', alpha=0.3)

plt.suptitle('Análisis comparativo: Ciudad vs Zonas de interés', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


## Relación ICS vs IPM por zona de interés


In [ ]:
# @title 30. Relación ICS vs IPM - scatter por zona
if len(gdf_obrero_filt) > 0 and len(gdf_ipm_obrero) > 0:
    merge_obrero = gdf_obrero_filt.merge(
        gdf_ipm_obrero[['COD_DANE', 'ipm']], left_on='cod_dane_a', right_on='COD_DANE', how='inner')
if len(gdf_roosevelt_filt) > 0 and len(gdf_ipm_roosevelt) > 0:
    merge_roosevelt = gdf_roosevelt_filt.merge(
        gdf_ipm_roosevelt[['COD_DANE', 'ipm']], left_on='cod_dane_a', right_on='COD_DANE', how='inner')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

for ax, data, nombre, color in [
    (ax1, merge_obrero if len(gdf_obrero_filt) > 0 else None, 'Barrio Obrero', 'blue'),
    (ax2, merge_roosevelt if len(gdf_roosevelt_filt) > 0 else None, 'Roosevelt', 'red')
]:
    if data is not None and len(data) > 0:
        x, y = data['ics'], data['ipm']
        ax.scatter(x, y, alpha=0.6, color=color, edgecolors='black', linewidth=0.5)
        coeffs = np.polyfit(x, y, 1)
        x_line = np.linspace(x.min(), x.max(), 100)
        ax.plot(x_line, coeffs[0] + coeffs[1] * x_line, color='black', linewidth=2, linestyle='--')
        corr = x.corr(y)
        ax.set_title(f'{nombre}  |  Correlación: {corr:.3f}', fontweight='bold')
        ax.text(0.05, 0.95, f'n = {len(data)}', transform=ax.transAxes,
                fontsize=11, va='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    else:
        ax.text(0.5, 0.5, 'Sin datos', ha='center', va='center', fontsize=14)
    ax.set_xlabel('Índice de Condición Social (ICS)')
    ax.set_ylabel('Índice de Pobreza Multidimensional (IPM)')
    ax.grid(alpha=0.3)

plt.suptitle('Relación ICS vs IPM por zona de interés', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()
